<a href="https://colab.research.google.com/github/dksgustn155-cmd/AIFFEL_quest_eng/blob/main/Day1_RAG_Code_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ 필수 라이브러리 한 번에 설치하기 (충돌 및 성능 최적화)
이 셀은 `pydantic` 버전 충돌을 해결하고 RAG 구현에 필요한 최신 패키지들을 설치합니다. 실행 후 **[Restart Session]** 버튼이 나타나면 반드시 클릭해 주세요.

In [ ]:
# 의존성 충돌 해결 및 필수 패키지 설치
!pip install -q -U pydantic==2.12.3
!pip install -q -U langchain langchain-community langchain-openai langchain-chroma langchain-huggingface langchain-text-splitters
!pip install -q -U pypdf tiktoken chromadb

### 🛠️ 필수 라이브러리 한 번에 설치하기 (충돌 방지)
이 셀을 실행한 후, 만약 하단에 'Restart Session' 버튼이 나오면 **한 번만 클릭**해 주세요. 그 후에는 다시 설치할 필요 없이 다음 단계로 넘어가시면 됩니다.

In [ ]:
# 의존성 충돌을 피하기 위해 필요한 패키지들을 특정 버전으로 한 번에 설치합니다.
!pip install -q -U langchain langchain-community langchain-openai langchain-core langchain-text-splitters langchain-chroma langchain-huggingface
!pip install -q -U pypdf tiktoken sentence_transformers chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 34.5 MB/s eta 0:00:00


last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [ ]:
import os


In [ ]:
#! curl ipinfo.io

In [ ]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    openai_api_key=userdata.get('안현') # API 키를 여기에 추가합니다.
)

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.


### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [ ]:
!pip install pydantic==2.12.3
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

  Using cached pydantic-2.12.3-py3-none-any.whl.metadata (87 kB)
  Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached pydantic-2.12.3-py3-none-any.whl (462 kB)
Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-client 0.44.0 requires pydantic>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
google-adk 1.29.0 requires opentelemet

/tmp/ipykernel_23884/2768883480.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [ ]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [ ]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [ ]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

print(documents[0].page_content)





































모두의연구소 ‘AI학교 아이펠’, ICLR 2024 워크숍 혁신 기술 논문 채택 < 일반 < 기업 < 기사본문 - IT조선











 






















































주요서비스 바로가기
본문 바로가기
매체정보 바로가기
로그인 바로가기
기사검색 바로가기
전체서비스 바로가기




























 






전체메뉴



기사검색








기사검색


검색

닫기














기사검색









기사검색


검색

닫기











로그인




facebook

post
youtube



UPDATED. 2026-05-26 15:59 (화) 




기사검색









기업

일반
모바일·가전
방송·통신
반도체·디스플레이
SW·보안
중공업·에너지
소부장·스타트업
유통·쇼핑
프롭테크·부동산



모빌리티

자동차·모빌리티
로봇·드론·항공
방산



게임·콘텐츠

게임·인터넷
메타버스·VR
키덜트
미디어·엔터



과학·헬스

과학·우주
의학·정책
제약·바이오



파이낸스

금융
증권
핀테크·블록체인



칼럼·인터뷰

칼럼
기고
인터뷰



알림

알립니다
인사
부음
보도자료



컴퓨팅·AI

일반
코딩
에듀테크
테크리포트



GLOBAL


GLOBAL
기획·연재
속보
백과사전











속보




기업


일반


모바일·가전


방송·통신


반도체·디스플레이


SW·보안


중공업·에너지


소부장·스타트업


유통·쇼핑


프롭테크·부동산




모빌리티


자동차·모빌리티


로봇·드론·항공


방산




게임·콘텐츠


게임·인터넷


메타버스·VR


키덜트


미디어·엔터




과학·헬스


과학·우주


의학·정책


제약·바이오




파이낸스


금융


증권


핀테크·블록체인




칼럼·인터뷰


칼럼


기고

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [ ]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [ ]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [ ]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [ ]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [ ]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [ ]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [ ]:
from google.colab import userdata

# '안현' 이름으로 저장된 API 키를 사용하여 클라이언트를 초기화합니다.
client = openai.OpenAI(api_key=userdata.get('안현'))

In [ ]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata

# api_key 인자에 저장된 API 키를 직접 전달하여 오류를 해결합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=userdata.get('안현')
)

#### 🚀 안정적인 OpenAI 임베딩 설정
`HuggingFaceEmbeddings`는 로컬 CPU 리소스를 많이 사용하여 중단될 가능성이 높습니다. 유료 API를 사용 중이시라면 `OpenAIEmbeddings`를 사용하는 것이 훨씬 빠르고 정확합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata

# OpenAI 임베딩 모델 설정
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=userdata.get('안현')
)
print("OpenAI 임베딩 모델 설정 완료")

OpenAI 임베딩 모델 설정 완료


만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [ ]:
!curl ipinfo.io

{
  "ip": "35.204.169.213",
  "hostname": "213.169.204.35.bc.googleusercontent.com",
  "city": "Groningen",
  "region": "Groningen",
  "country": "NL",
  "loc": "53.2192,6.5667",
  "org": "AS396982 Google LLC",
  "postal": "9711",
  "timezone": "Europe/Amsterdam",
  "readme": "https://ipinfo.io/missingauth"
}

In [ ]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

!pip install -q sentence_transformers langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings

# 최신 패키지인 langchain_huggingface를 사용하여 지역 제한 문제를 우회합니다.
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [ ]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [ ]:
print(embeddings[1])

[-0.043598730117082596, 0.046768199652433395, -0.026038166135549545, -0.0013704403536394238, 0.06055562198162079, 0.05017256364226341, 0.11730732768774033, -0.027721069753170013, 0.009279526770114899, 0.07751473039388657, 0.001512883580289781, -0.09486886113882065, 0.0018310161540284753, 0.010358039289712906, 0.02260895073413849, 0.043929990381002426, -0.013756091706454754, -0.020579246804118156, -0.0327138714492321, -0.02561490423977375, 0.043547168374061584, 0.08289312571287155, -0.05014246329665184, -0.04123270511627197, -0.05022479221224785, 0.04015067592263222, 0.05747781693935394, 0.012200281023979187, 0.028081471100449562, -0.0607445202767849, -0.0662989392876625, 0.02342132292687893, 0.04859095811843872, -0.015047372318804264, -0.020352834835648537, -0.003334973705932498, 0.03370742127299309, -0.10650115460157394, 0.034283142536878586, 0.06269633769989014, 0.03697563335299492, 0.039933156222105026, 0.060280971229076385, -0.052957531064748764, -0.02507835440337658, 0.06646721810

In [ ]:
len(embeddings[1])

384

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [ ]:
query = ["this is red fruit"]

In [ ]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7799129958370878
0.602116871556071
0.5388708787265135


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [ ]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [ ]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [ ]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [ ]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [ ]:
print(docs[0].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [ ]:
!pip install -U langchain langchain-classic

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [ ]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI
from google.colab import userdata

# OpenAI 모델 설정: API 키를 명시적으로 전달하여 오류를 해결합니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    streaming=True,
    openai_api_key=userdata.get('안현'),
    callbacks=[StreamingStdOutCallbackHandler()],
)
print("ChatOpenAI 설정 완료")

ChatOpenAI 설정 완료


체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [ ]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [ ]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_23884/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is somewhat wild-looking, with an interrogative and erratic expression. His features are wayward and obstinate, yet his mouth has something soft and childlike about it. The masculinity and strength are in his eyes and brow, while the lower half of his face is tender, immature, and somewhat feminine. His chin is described as irresolute and boylike, contradicting the forehead and expression. His dark brown eyes are full of pride and humility.

마크다운 형식으로 출력해봅니다

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is somewhat wild-looking, with an interrogative and erratic expression. His features are wayward and obstinate, yet his mouth has something soft and childlike about it. The masculinity and strength are in his eyes and brow, while the lower half of his face is tender, immature, and somewhat feminine. His chin is described as irresolute and boylike, contradicting the forehead and expression. His dark brown eyes are full of pride and humility.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [ ]:
from google.colab import userdata

# API 키를 명시적으로 전달하여 인증 오류를 해결합니다.
llm2 = ChatOpenAI(
    model="gpt-4o",
    openai_api_key=userdata.get('안현')
)

request = llm2.invoke("how demian looks like")
display(Markdown(request.content))

If you're referring to Demian from Hermann Hesse's novel "Demian," there's no definitive description of his physical appearance provided in the text, as the focus is more on his character and the symbolic role he plays in the protagonist Emil Sinclair's life. Demian is often portrayed as having an enigmatic and magnetic presence, which influences Sinclair's personal and spiritual development.

However, in adaptations or visual interpretations, Demian is often depicted as a young man with an intense, insightful gaze that reflects his wisdom and maturity beyond his years. His appearance might be somewhat refined or aristocratic, hinting at his role as a mentor or guide in Sinclair's journey towards self-discovery. The emphasis is typically on his compelling aura and the intellectual and philosophical impact he has on those around him, rather than specific physical traits.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [ ]:
# 1. 필수 라이브러리 설치 및 의존성 해결
!pip install -q -U pydantic==2.12.3
!pip install -q -U langchain langchain-community langchain-openai langchain-chroma langchain-huggingface langchain-text-splitters
!pip install -q -U pypdf tiktoken chromadb sentence_transformers

print("모든 라이브러리 설치가 완료되었습니다. 세션을 다시 시작해야 할 수도 있습니다.")

모든 라이브러리 설치가 완료되었습니다. 세션을 다시 시작해야 할 수도 있습니다.


Text splitter 사용을 위한 준비입니다

In [ ]:
import tiktoken

# 토큰 길이를 계산하는 함수 정의
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

print("토큰 계산 함수 준비 완료")

토큰 계산 함수 준비 완료


### Step 1 Document loader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 로드
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

print(f"문서 로드 완료: 총 {len(pages)} 페이지")

문서 로드 완료: 총 182 페이지


### Step 2 Text splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할 설정 (토큰 기준으로 의미 단위 분할)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len
)

docs = text_splitter.split_documents(pages)
print(f"텍스트 분할 완료: {len(docs)} 개의 청크 생성")

텍스트 분할 완료: 182 개의 청크 생성


### Step 3 Vector Empeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata

# Step 3: OpenAI 임베딩 모델 설정
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=userdata.get('안현')
)

print("임베딩 모델 설정 완료")

임베딩 모델 설정 완료


In [ ]:
from langchain_chroma import Chroma

# VectorStore 생성: 차원 충돌을 피하기 위해 새로운 collection_name을 지정합니다.
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="demian_openai_final"
)

print("VectorStore 생성 및 인덱싱 완료 (OpenAI 1536차원)")

VectorStore 생성 및 인덱싱 완료 (OpenAI 1536차원)


### Step 4 Retrievers

In [ ]:
# Step 4: Retriever 설정 (MMR 방식으로 다양성 확보)
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

print("Retriever 설정 완료")

Retriever 설정 완료


In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_openai import ChatOpenAI

# Step 5를 위한 LLM 및 RetrievalQA 체인 준비
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    openai_api_key=userdata.get('안현')
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("Q&A 체인 준비 완료")

Q&A 체인 준비 완료


### Step 5 Question Answering

In [ ]:
# 한글로 질문해도 GPT가 영어 문서를 검색하여 한글로 답변해 줍니다.
query = "데미안의 외모는 어떻게 묘사되어 있니?"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 데미안의 외모는 어떻게 묘사되어 있니?


데미안의 외모는 다음과 같이 묘사되어 있습니다. 그는 봄처럼 가볍고 곧은 자세로 걸어오며, 그의 얼굴은 지적이고 결단력 있는 입과 넓은 이마의 독특한 광채를 가지고 있습니다. 그는 담황색의 레인코트를 입고 얇은 지팡이를 팔에 걸고 있습니다.

In [ ]:
query = "스토리 요약해줘"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 스토리 요약해줘


"Demian"은 헤르만 헤세의 소설로, 주인공 싱클레어가 자신의 내면 세계를 탐구하고 성장하는 과정을 그립니다. 싱클레어는 어린 시절부터 선과 악의 경계에서 혼란을 겪으며, 자신만의 길을 찾기 위해 노력합니다. 그는 데미안이라는 신비로운 친구를 만나게 되며, 데미안은 싱클레어에게 새로운 시각과 깨달음을 제공합니다. 싱클레어는 데미안과의 만남을 통해 자신의 정체성을 찾고, 사회의 규범을 넘어선 새로운 삶의 의미를 발견하게 됩니다. 이 과정에서 그는 내면의 갈등과 외부 세계와의 충돌을 경험하며, 진정한 자아를 찾아가는 여정을 그립니다.

In [ ]:
query = "데미안은 악인이야?"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 데미안은 악인이야?


주어진 문맥에서는 데미안이 악인이라는 직접적인 언급은 없습니다. 데미안은 주인공인 싱클레어에게 중요한 인물로, 그의 삶에 영향을 미치는 존재로 묘사됩니다. 데미안은 싱클레어에게 확신과 평온을 주는 인물로 나타나며, 그와의 재회는 싱클레어에게 안도감을 줍니다. 따라서 데미안을 악인으로 단정짓기는 어렵습니다.

In [ ]:
query = "그렇다면 싱클레어의 생김새에 대해서 알려줘"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 그렇다면 싱클레어의 생김새에 대해서 알려줘


죄송하지만, 주어진 문맥에서는 싱클레어의 생김새에 대한 구체적인 설명이 없습니다.

In [ ]:
query = "왜 싱클레어에 대한 구체적인 설명이 없는거야"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 왜 싱클레어에 대한 구체적인 설명이 없는거야


싱클레어에 대한 구체적인 설명이 없는 이유는 주어진 문맥에서 그의 내면적 경험과 감정에 더 중점을 두고 있기 때문입니다. 이 문맥에서는 싱클레어가 겪는 감정적 변화와 그의 삶에 영향을 미치는 사건들, 예를 들어 데미안과의 대화나 비아트리체에 대한 감정 등이 강조되고 있습니다. 이러한 요소들은 싱클레어의 성장을 보여주기 위한 중요한 부분으로, 그의 외적인 특징보다는 내적인 여정에 초점을 맞추고 있습니다.

In [ ]:
query = "이 이야기의 결말을 얘기해줘"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 이 이야기의 결말을 얘기해줘


죄송하지만, 주어진 정보로는 이야기의 결말을 알 수 없습니다. "데미안"의 결말에 대해 더 알고 싶다면, 책을 직접 읽어보시거나 관련 자료를 찾아보시는 것을 추천드립니다.

In [ ]:
query = "그럼 몇막으로 구성되어있어?"
response = qa_chain.invoke(query)

from IPython.display import Markdown, display
print("질문:", query)
display(Markdown(response["result"]))

질문: 그럼 몇막으로 구성되어있어?


죄송하지만, 제공된 정보로는 작품이 몇 막으로 구성되어 있는지 알 수 없습니다.